# eo-monitor walkthrough

**Question:** How far did peak-season vegetation/moisture over an eastern Nebraska AOI depart from the 2019–2022 normal during the 2023 Corn Belt flash drought?

The index and anomaly math run offline with only `numpy`. The full STAC → cube → COG flow is shown as code you can execute once the geospatial environment is installed (`pixi install`).

## 1. Load and inspect the config

Everything is config-driven. The flagship config targets the drought peak (Jul–Aug 2023) against a 2019–2022 July/August baseline.

In [ ]:
from eo_monitor.config import load_config

cfg = load_config("../config/corn_belt.yaml")
print("AOI bbox:", cfg.aoi.bbox)
print("Target:  ", cfg.date_range.as_query())
print("Baseline:", cfg.baseline.as_query(), "months", cfg.baseline.months)
print("Indices: ", cfg.indices)

## 2. The index math is pure and hand-verifiable

NDVI = (NIR − Red)/(NIR + Red), NDWI = (Green − NIR)/(Green + NIR), NDMI = (NIR − SWIR)/(NIR + SWIR).

In [ ]:
import numpy as np

from eo_monitor.indices import ndvi

nir = np.array([0.5, 0.8, 0.2])
red = np.array([0.1, 0.2, 0.2])
print("NDVI:", ndvi(nir, red))  # -> [0.666..., 0.6, 0.0]

## 3. Anomaly = standardised departure from the baseline

`z = (value − baseline_mean) / baseline_std`, computed per pixel over the baseline window.

In [ ]:
from eo_monitor.anomaly import anomaly_cube

baseline = np.array([[0.0], [2.0], [4.0], [6.0]])  # one pixel, 4 times -> mean 3, std sqrt(5)
target = np.array([3.0 + np.sqrt(5.0)])  # exactly +1 sigma
print("z:", anomaly_cube(target, baseline))  # -> [1.0]

## 6. The full pipeline (requires the geospatial environment)

The cell below is the real run. It is left unexecuted here because it needs the conda-forge stack (GDAL, rasterio, odc-stac) and network access to the STAC catalogue. From a clean clone: `pixi install` then run it, or use the CLI: `pixi run eo-monitor run --config config/corn_belt.yaml`.

In [ ]:
from eo_monitor.anomaly import classify_anomaly, robust_zscore
from eo_monitor.indices import evi2, nbr, savi

nir = np.array([0.5, 0.45])
red = np.array([0.1, 0.06])
print("SAVI (L=0.5):", savi(nir, red))
print("EVI2:        ", evi2(nir, red))
print("NBR:         ", nbr(np.array([0.6, 0.4]), np.array([0.2, 0.4])))

# Robust z-score shrugs off a single extreme baseline value.
baseline = np.array([[1.0], [2.0], [4.0], [100.0]])  # median 3, MAD = 1.4826*1.5
z = robust_zscore(np.array([3.0 + 1.4826 * 1.5]), baseline)
print("robust z:", z, "-> exactly +1 sigma")
print("classify:", classify_anomaly(np.array([-3.0, 0.0, 2.5])))  # [-1, 0, 1]

## 5. End-to-end demo with real numbers (offline)

`run_demo` synthesises a small seeded reflectance cube with a planted vegetation-loss patch, runs the real index + anomaly code, and recovers the patch as a |z| > 2 anomaly. It needs only numpy and runs in well under a second.

In [ ]:
from eo_monitor.demo import run_demo

metrics = run_demo(seed=0, out_dir="../outputs")
for k, v in metrics.items():
    print(f"{k:24s} {v}")
# planted_region_recovered == 1.0: the |z| > 2 loss mask recovers the whole patch.

## 4. The full pipeline (requires the geospatial environment)

The cell below is the real run. It is left unexecuted here because it needs the conda-forge stack (GDAL, rasterio, odc-stac) and network access to the STAC catalogue. From a clean clone: `pixi install` then run it, or use the CLI: `pixi run eo-monitor run --config config/corn_belt.yaml`.

In [ ]:
# from eo_monitor.catalog import search_for_config
# from eo_monitor.cube import load_cube
# from eo_monitor.indices import compute_index
# from eo_monitor.anomaly import anomaly_cube
# from eo_monitor import io as eo_io
#
# target_items = search_for_config(cfg)
# baseline_items = search_for_config(cfg, window=cfg.baseline)
# target_cube = load_cube(target_items, cfg)
# baseline_cube = load_cube(baseline_items, cfg)
#
# tb = {b: target_cube[b] for b in target_cube.data_vars if b != 'SCL'}
# bb = {b: baseline_cube[b] for b in baseline_cube.data_vars if b != 'SCL'}
# ndvi_target = compute_index('NDVI', tb).median(dim='time', skipna=True)
# ndvi_baseline = compute_index('NDVI', bb)
# z = anomaly_cube(ndvi_target, ndvi_baseline, dim='time')
# eo_io.write_cog(z, 'outputs/ndvi_anomaly.tif')
# eo_io.write_quicklook(z, 'outputs/ndvi_anomaly.png', vmin=-3, vmax=3)